# SummarizationMetric

## What it measures

Two things at once, and takes the worse of them:

- **Alignment** - does the summary contain anything the source text does not support?
  (hallucination within a summary)
- **Coverage** - does the summary answer the questions the source text can answer?

DeepEval generates `n` assessment questions from the source, answers them from both texts,
and compares. The final score is the minimum of the two sub-scores, so a fluent summary
that quietly drops half the source cannot pass on style.

## When it is useful

Any place a long document is compressed for a human who will act on the compression
without reading the original - exactly the position a compliance analyst is in when
reading an AI-generated policy summary. Coverage failures here are the dangerous ones: the
omitted clause is the one nobody knows was omitted.

## DeepEval inputs and test-case type

| DeepEval field | Required |
|---|---|
| test case type | `LLMTestCase` |
| `input` | yes - **the source text being summarized**, not a user question |
| `actual_output` | yes - the summary |

Note the unusual convention: for this metric `input` is the original document. Passing a
question there, as every other single-turn metric expects, silently measures nothing
useful.

In [ ]:
# --------------------------------------------------------------------------
# Configuration. Every value comes from the environment - nothing about this
# machine, this port or this deployment is baked into the notebook.
# --------------------------------------------------------------------------
import json
import os
import textwrap
from pathlib import Path

import httpx
from dotenv import load_dotenv

# Look for .env next to the notebook, then one level up (the project root).
for _candidate in (Path.cwd() / ".env", Path.cwd().parent / ".env"):
    if _candidate.is_file():
        load_dotenv(_candidate)
        break


class MissingConfiguration(RuntimeError):
    """Raised when a required environment variable is absent."""


def env(name, default=None, *, required=False):
    value = os.environ.get(name) or default
    if required and not value:
        raise MissingConfiguration(
            f"Environment variable {name!r} is not set.\n"
            f"Copy .env.example to .env and fill it in, or export {name} before "
            f"starting the kernel. See README.md -> '.env configuration'."
        )
    return value


API_BASE = env("AML_API_BASE_URL", "http://localhost:8000").rstrip("/")
API_TIMEOUT_S = float(env("AML_API_TIMEOUT_S", "180"))
EXPECTED_SEED_VERSION = env("AML_EXPECTED_SEED_VERSION", "scenarios-v1")
RESET_BEFORE_RUN = env("AML_RESET_BEFORE_RUN", "false").lower() in ("1", "true", "yes")

# Every notebook needs an OpenAI key. ToolCorrectnessMetric scores without any
# LLM call, but DeepEval 4.1.4 still builds a GPTModel in its constructor and
# raises without a key, so the key is required there too - just never used.
JUDGE_MODEL = env("DEEPEVAL_JUDGE_MODEL", "gpt-5.4-mini")
os.environ.setdefault("DEEPEVAL_TELEMETRY_OPT_OUT", "YES")

# One key per role, falling back to the single AML_API_KEY. Blank is correct
# when the application runs with AUTH_MODE=off (its default).
API_KEYS = {
    "analyst": env("AML_API_KEY_ANALYST") or env("AML_API_KEY", ""),
    "eval_reader": env("AML_API_KEY_EVAL_READER") or env("AML_API_KEY", ""),
    "test_operator": env("AML_API_KEY_TEST_OPERATOR") or env("AML_API_KEY", ""),
}

print(f"API base URL       : {API_BASE}")
print(f"Request timeout    : {API_TIMEOUT_S}s")
print(f"Judge model        : {JUDGE_MODEL}")
print(f"Expected seed      : {EXPECTED_SEED_VERSION}")
print(f"API key configured : {bool(API_KEYS['analyst'])}  (False is correct when AUTH_MODE=off)")
print(f"OPENAI_API_KEY set : {bool(os.environ.get('OPENAI_API_KEY'))}")

In [ ]:
# --------------------------------------------------------------------------
# A small HTTP client. Every failure mode the application can present is
# turned into a message that names the cause and the thing to check.
# --------------------------------------------------------------------------
# The contract these notebooks were written against. The application may
# serve a HIGHER minor version: a MINOR bump is additive by its own
# contract policy (1.0.0 -> 1.1.0 added HealthResponse.build_version and
# changed nothing else), so treating it as a mismatch would turn this
# guard into noise on every single call. Only a MAJOR change, or an
# application older than these notebooks, is a problem.
EXPECTED_SCHEMA_VERSION = "1.0.0"


def contract_version(value):
    """(major, minor) from a MAJOR.MINOR.PATCH string, or None."""
    try:
        parts = value.split("+")[0].split(".")
        return int(parts[0]), int(parts[1])
    except (AttributeError, IndexError, ValueError):
        return None

SECRET_KEY_HINTS = ("api_key", "apikey", "authorization", "secret", "password",
                    "credential", "token")


def redact(value):
    """Mask credential-like values before anything is printed."""
    if isinstance(value, dict):
        return {
            k: ("***REDACTED***" if any(h in k.lower() for h in SECRET_KEY_HINTS)
                else redact(v))
            for k, v in value.items()
        }
    if isinstance(value, list):
        return [redact(v) for v in value]
    return value


class ApiError(RuntimeError):
    """A non-2xx response, carrying the application's error envelope."""


def api(method, path, *, role="analyst", json_body=None, params=None,
        expect_status=None):
    """Call the application API and return parsed JSON.

    role selects which API key is sent. It only matters when the application
    runs with AUTH_MODE=api_key; with AUTH_MODE=off the header is omitted.
    """
    headers = {"Accept": "application/json"}
    key = API_KEYS.get(role, "")
    if key:
        headers["X-API-Key"] = key

    url = f"{API_BASE}{path}"
    try:
        response = httpx.request(method, url, headers=headers, json=json_body,
                                 params=params, timeout=API_TIMEOUT_S)
    except httpx.ConnectError as exc:
        raise ApiError(
            f"Could not connect to {url}.\n"
            f"  - Is the application running?  curl {API_BASE}/api/health\n"
            f"  - Is AML_API_BASE_URL correct? It is currently {API_BASE!r}.\n"
            f"  - Underlying error: {exc}"
        ) from exc
    except httpx.TimeoutException as exc:
        raise ApiError(
            f"{method} {url} timed out after {API_TIMEOUT_S}s.\n"
            f"  - An investigation run does retrieval, several MCP tool calls and\n"
            f"    one LLM synthesis; raise AML_API_TIMEOUT_S if this is expected.\n"
            f"  - Underlying error: {exc!r}"
        ) from exc

    served = response.headers.get("X-Schema-Version")
    served_version = contract_version(served) if served else None
    expected_version = contract_version(EXPECTED_SCHEMA_VERSION)
    if served_version and served_version[0] != expected_version[0]:
        raise ApiError(
            f"The application serves contract version {served}; these notebooks were "
            f"written against {EXPECTED_SCHEMA_VERSION}. A MAJOR change means fields "
            f"may have been removed or retyped - re-derive the goldens against the "
            f"new contract rather than scoring against one they do not match."
        )
    if served_version and served_version[1] < expected_version[1]:
        print(f"WARNING: application reports contract version {served}, older than "
              f"the {EXPECTED_SCHEMA_VERSION} these notebooks were written against. "
              f"Fields the goldens rely on may not exist yet.")

    if response.status_code >= 400:
        try:
            envelope = response.json()
        except ValueError:
            envelope = {"raw_body": response.text[:1000]}
        hint = {
            401: "AUTH_MODE=api_key is on and no valid X-API-Key was sent. Set AML_API_KEY.",
            403: "The key's role may not reach this endpoint. eval_reader is needed for "
                 "/api/agent/trace and /api/eval/*; test_operator for /api/dev/reset and "
                 "/api/mcp/invoke.",
            404: "The id does not exist. Resolve ids from GET /api/eval/scenarios rather "
                 "than hardcoding them.",
            409: "Often index_not_built - the vector index has never been built. "
                 "POST /api/dev/reset once, or set AML_RESET_BEFORE_RUN=true.",
            502: "The application's LLM provider failed or returned output that broke its "
                 "own schema contract. Retry, or inspect GET /api/agent/trace/{run_id}.",
            503: "llm_not_configured - the application has no OPENROUTER_API_KEY. "
                 "This is the application's key, not the judge's OPENAI_API_KEY.",
        }.get(response.status_code, "")
        raise ApiError(
            f"{method} {url} -> HTTP {response.status_code}\n"
            f"  envelope: {json.dumps(envelope, indent=2)[:1200]}\n"
            + (f"  hint: {hint}" if hint else "")
        )

    if expect_status is not None and response.status_code != expect_status:
        raise ApiError(f"{method} {url} -> expected HTTP {expect_status}, "
                       f"got {response.status_code}")

    if not response.content:
        return None
    try:
        return response.json()
    except ValueError as exc:
        raise ApiError(
            f"{method} {url} returned HTTP {response.status_code} but the body is not "
            f"JSON.\n  first 500 bytes: {response.text[:500]!r}"
        ) from exc


def show(title, payload, limit=2500):
    """Pretty-print a payload with secrets masked and long bodies truncated."""
    text = json.dumps(redact(payload), indent=2, default=str)
    print(f"----- {title} -----")
    print(text if len(text) <= limit else text[:limit] + f"\n... [{len(text) - limit} more characters]")


health = api("GET", "/api/health")
show("GET /api/health", health)
if not health.get("status") == "ok":
    raise ApiError(f"Application is not healthy: {health}")

## Endpoint exercised

- `GET /api/documents/{document_id}` supplies the **source text**: the complete markdown of
  policy AML-001, exactly as the application serves it.
- `POST /api/rag/query` is asked to summarize that policy, producing the **summary**.

This pairing is what makes the metric meaningful black-box. The source is a document the
application publishes in full, so the reference side of the comparison is authoritative and
needs no hand-authored golden; the summary is real generated output from the endpoint under
test.

`include_history: false` and no `case_id` keep the run stateless: a caseless query has no
conversation thread at all, so nothing from a previous run can leak into the prompt.

In [ ]:
# --------------------------------------------------------------------------
# Resolve scenarios to live row ids. Seed ids are assigned by insert order, so
# a hardcoded case_id silently rebinds to a different case when the seed data
# changes. GET /api/eval/scenarios exists precisely to avoid that.
# --------------------------------------------------------------------------
if RESET_BEFORE_RUN:
    # Drops and recreates every table, restoring deterministic seed state.
    reset = api("POST", "/api/dev/reset", role="test_operator")
    show("POST /api/dev/reset", reset)

SCENARIOS = {s["scenario_id"]: s for s in api("GET", "/api/eval/scenarios",
                                              role="eval_reader")}

seed_versions = {s["seed_version"] for s in SCENARIOS.values()}
if seed_versions != {EXPECTED_SEED_VERSION}:
    raise RuntimeError(
        f"Seed version mismatch: application reports {seed_versions}, the goldens in "
        f"this notebook were authored against {EXPECTED_SEED_VERSION!r}.\n"
        f"A golden authored against different seed data is not a weaker test, it is a "
        f"wrong one - fix the seed or the golden rather than lowering the threshold."
    )

for sid, s in sorted(SCENARIOS.items()):
    print(f"{sid}: case_id={s['case_id']} customer_id={s['customer_id']} "
          f"transaction_id={s['transaction_id']}  {s['title']}")

In [ ]:
# --------------------------------------------------------------------------
# The exact request.
#
# The question names the policy explicitly so retrieval has a strong lexical
# and semantic anchor. top_k is raised to 10 because a summary needs breadth -
# a summarization metric run over an answer built from two chunks is really
# measuring the retriever's top_k, not the summarizer.
# --------------------------------------------------------------------------
QUESTION = (
    "Summarize the cash transaction thresholds and the structuring indicators set out "
    "in transaction monitoring policy AML-001."
)

request_body = {"question": QUESTION, "top_k": 10, "include_history": False}

print("POST", f"{API_BASE}/api/rag/query")
print("headers:", json.dumps(redact({"X-API-Key": API_KEYS["analyst"] or None,
                                     "Content-Type": "application/json"}), indent=2))
print("body:", json.dumps(request_body, indent=2))

In [ ]:
# --------------------------------------------------------------------------
# The raw response.
# --------------------------------------------------------------------------
response = api("POST", "/api/rag/query", json_body=request_body)

show("POST /api/rag/query", {k: v for k, v in response.items()
                             if k != "retrieved_context"})
print()
print("SUMMARY (actual_output)")
print(textwrap.fill(response["answer"], width=96, initial_indent="  ",
                    subsequent_indent="  "))
print()
print("chunks the summary was built from:")
for chunk in response["retrieved_context"]:
    print(f"  {chunk['chunk_id']:<12} score={chunk['score']:.3f} "
          f"section={chunk['metadata'].get('section')!r}")

## Mapping the API response onto DeepEval fields

| DeepEval field | Source | Note |
|---|---|---|
| `input` | `content` of AML-001 from `GET /api/documents/{id}` | The source text, **not** the question |
| `actual_output` | `answer` from `POST /api/rag/query` | The summary |

The question itself (`QUESTION`) is not passed to the metric at all - it is an instruction
to the application, not part of the summarization comparison. It is printed above so a
failure can be traced back to a badly framed request.

In [ ]:
# --------------------------------------------------------------------------
# Deriving the source text.
#
# No golden is authored by hand here. The reference is the policy document the
# application itself serves, which is the strongest possible ground truth for
# a black-box test: if the document changes, the test changes with it.
#
# The assertion confirms the document actually contains the material the
# question asked about, so a coverage failure means the summarizer omitted
# something rather than that the notebook pointed at the wrong document.
# --------------------------------------------------------------------------
policy_index = api("GET", "/api/documents", params={"type": "policy"})
AML001 = next((d for d in policy_index if "AML-001" in (d.get("source") or "")), None)
if AML001 is None:
    raise RuntimeError(
        "No document with source 'Internal Compliance Policy AML-001' is served by "
        "GET /api/documents?type=policy. Available: "
        + ", ".join(repr(d.get("source")) for d in policy_index)
    )

document = api("GET", f"/api/documents/{AML001['document_id']}")
SOURCE_TEXT = document["content"]

print(f"document_id  : {document['document_id']}")
print(f"source       : {document.get('source')!r}")
print(f"version_date : {document.get('version_date')}")
print(f"length       : {len(SOURCE_TEXT)} characters")
print()

REQUIRED_TOPICS = ["10,000", "25,000", "structuring"]
absent = [t for t in REQUIRED_TOPICS if t.lower() not in SOURCE_TEXT.lower()]
if absent:
    raise RuntimeError(
        f"AML-001 as served does not mention {absent}. The question asks about material "
        f"the source does not contain, so a low coverage score would be the notebook's "
        f"fault, not the application's. Update the question or the document choice."
    )
print("Source text (first 900 characters):")
print(textwrap.indent(SOURCE_TEXT[:900], "  "))

In [ ]:
# --------------------------------------------------------------------------
# Build the test case and print each DeepEval role explicitly.
# --------------------------------------------------------------------------
from deepeval.test_case import LLMTestCase

test_case = LLMTestCase(
    input=SOURCE_TEXT,               # the source document - this metric's convention
    actual_output=response["answer"],  # the summary
)

print(f"USER INPUT (source text to be summarized): {len(test_case.input)} characters")
print(textwrap.indent(test_case.input[:400] + "...", "  "))
print()
print("ACTUAL OUTPUT (the summary under test)")
print(textwrap.fill(test_case.actual_output, width=96, initial_indent="  ",
                    subsequent_indent="  "))
print()
print("EXPECTED OUTPUT (golden): not used - the source document is the reference")

## Judge and threshold

- **Judge model**: `DEEPEVAL_JUDGE_MODEL`, default `gpt-5.4-mini`.
- **Threshold**: `0.5`, DeepEval's documented default for `SummarizationMetric`.
- **`n=5`**: DeepEval's default number of generated assessment questions, stated
  explicitly so the run is reproducible - changing `n` changes the coverage denominator
  and therefore the score.

The default threshold is kept. Because the final score is `min(alignment, coverage)`, a
pass at `0.5` guarantees only that neither half collapsed. That is the right bar for a
demonstration; a production gate should separate the two concerns and hold alignment much
higher than coverage, since inventing a requirement is worse than omitting one.

In [ ]:
from deepeval.metrics import SummarizationMetric

metric = SummarizationMetric(
    threshold=0.5,          # DeepEval's documented default
    n=5,                    # DeepEval's default number of assessment questions
    model=JUDGE_MODEL,
    include_reason=True,
    async_mode=False,
    verbose_mode=True,
)
print(f"metric class : {type(metric).__name__}")
print(f"judge model  : {JUDGE_MODEL}")
print(f"threshold    : {metric.threshold}")
print(f"async_mode   : {metric.async_mode}")
print(f"strict_mode  : {metric.strict_mode}")

In [ ]:
# --------------------------------------------------------------------------
# Run the metric. A judge failure is caught and explained rather than left as
# a bare traceback, because "the judge could not be reached" and "the
# application scored badly" are completely different findings.
# --------------------------------------------------------------------------
try:
    metric.measure(test_case)
except Exception as exc:                      # noqa: BLE001 - diagnostic wrapper
    message = str(exc)
    print(f"METRIC EXECUTION FAILED: {type(exc).__name__}: {message[:600]}")
    if "api_key" in message.lower() or "authentication" in message.lower():
        print("  -> OPENAI_API_KEY is missing or rejected. This is the judge's key, "
              "not the application's.")
    elif "model" in message.lower() and "not" in message.lower():
        print(f"  -> The judge model {JUDGE_MODEL!r} was rejected. Check that your "
              f"OpenAI account can reach it, and that the installed DeepEval version "
              f"knows the id. Set DEEPEVAL_JUDGE_MODEL to change it.")
    elif "rate" in message.lower():
        print("  -> Rate limited by the judge provider. Re-run the cell.")
    raise

In [ ]:
# --------------------------------------------------------------------------
# Score, verdict, reason and debug output.
#
# Read `metric.is_successful()`, never the raw score: DeepEval metrics do not
# all point the same way. AnswerRelevancy and ToolCorrectness are "higher is
# better"; Bias and Hallucination are rates where lower is better; PIILeakage
# is a privacy score where 0.0 means maximum leakage. is_successful() applies
# the correct comparison for the metric.
# --------------------------------------------------------------------------
print(f"metric          : {type(metric).__name__}")
print(f"judge model     : {JUDGE_MODEL}")
print(f"threshold       : {metric.threshold}")
print(f"score           : {metric.score}")
print(f"PASS / FAIL     : {'PASS' if metric.is_successful() else 'FAIL'}")
print(f"judge cost (USD): {metric.evaluation_cost}")
print()
print("reason:")
print(textwrap.fill(str(metric.reason), width=96, subsequent_indent="  "))
print()
print("----- verbose judge log (debug) -----")
print(metric.verbose_logs or "(none - construct the metric with verbose_mode=True)")

## Limitations in a black-box acceptance test

1. **Coverage is measured against the whole document, but the summarizer only saw
   retrieved chunks.** The application is a RAG endpoint: it answers from `top_k` chunks,
   not from the full policy. Any clause that was not retrieved is counted as a coverage
   miss even though the generator never had it. A low coverage score is therefore a
   *pipeline* result, not a generator verdict - check `retrieved_context` before blaming
   the model.
2. **The question shapes the score.** A narrow question produces a narrow summary that
   loses coverage against a broad document. The score is only comparable across runs when
   the question and `top_k` are held fixed, as they are here.
3. **Judge variance is visible on this metric.** In authoring runs the judge occasionally
   flagged a correctly summarized clause as unsupported. Treat one run as a signal, not a
   verdict.
4. **`n` is a hidden parameter.** Five generated questions is a small sample of what a
   document can be asked; two runs can legitimately differ because the questions differed.
5. **The investigation endpoint is a poor fit for this metric, despite looking like one.**
   `POST /api/cases/{id}/investigate` returns a `rationale` that reads like a summary of
   the case, but it is a *justification*: it deliberately cites only the evidence that
   drove the recommendation. Scored against the full case bundle it produces low coverage
   by design - during authoring it scored 0.40 for exactly that reason. That is the metric
   correctly describing a text that is not a summary, and it is why this notebook scores
   the policy-summarization flow instead.